
# TempoRun 2026 — Temporal Top-5 → Top-1 Reranker

Notebook độc lập, viết theo OOP, tái tạo pipeline đã nâng submission từ khoảng **68.9 lên 70.9**.

## Thuật toán

Với mỗi task:

1. Đọc top 10 từ `submission_68p9.json`.
2. Giữ nguyên rank 6–10.
3. Với từng candidate rank 1–5, lấy clip 5 frame theo thứ tự:
   `[-500, -250, 0, +250, +500] ms`.
4. Qwen3-VL-Reranker-2B chấm một temporal score cho mỗi clip.
5. Candidate temporal winner chỉ được đưa lên rank 1 khi đồng thời:
   - `winner_score - old_rank1_score >= 0.15`
   - `winner_score - runner_up_score >= 0.10`
6. Nếu không đủ ngưỡng, giữ nguyên thứ tự.
7. Nếu đủ ngưỡng, dùng **move-to-front**; không sort tự do toàn bộ top 5.
8. Không thay đổi `frame_ms`.

## Output

- `submission_68p9_temporal_top5.json`
- `submission_68p9_temporal_top5.zip`
- `temporal_top5_scores.csv`

Notebook mặc định dùng tối đa 2 GPU và chạy đầy đủ 300 task.


## 1. Cài thư viện — chỉ bật khi môi trường Kaggle chưa có phiên bản phù hợp

Đặt `INSTALL_DEPENDENCIES = True` rồi chạy cell. Sau khi cài, restart session nếu Kaggle yêu cầu.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys

    subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "sentence-transformers>=5.4.0",
        "transformers>=4.57.0",
        "accelerate",
        "bitsandbytes",
        "qwen-vl-utils",
        "tqdm",
        "opencv-python-headless",
        ]
    )
    

## 2. Cấu hình

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TemporalRerankConfig:
    # -------------------------
    # Kaggle input
    # -------------------------
    input_submission: Path = Path(
        "submission_final_8B_rerank500_10.json"
    )

    tasks_jsonl_candidates: tuple[Path, ...] = (
        Path(
            "private_round_tasks.jsonl"
        ),
        Path(
            "private_round_tasks.jsonl"
        ),
    )

    video_dataset_root: Path = Path(
        "V3C"
    )

    reranker_model: Path = Path(
        "Qwen3-VL-Reranker-8B"
    )

    # -------------------------
    # Output
    # -------------------------
    output_dir: Path = Path(
        "temporal_top5_rerank_final_private_task"
    )

    # -------------------------
    # Temporal clip
    # -------------------------
    temporal_offsets_ms: tuple[int, ...] = (
        -1000,
        -500,
        0,
        500,
        1000,
    )

    # Tạo nhiều center xung quanh frame_ms gốc.
    local_shifts_ms: tuple[int, ...] = (
        -250,
        0,
        250,
    )

    # Phạt các center nằm xa timestamp gốc.
    distance_penalty: float = 0.025
    
    max_frame_side: int = 640

    # -------------------------
    # Ranking
    # -------------------------
    top_k: int = 5
    final_top: int = 10
    promote_over_old_top1: float = 0.15
    winner_over_runner_up: float = 0.10

    # -------------------------
    # Local shot-cut guard
    # -------------------------
    enable_cut_guard: bool = True
    cut_hist_correlation: float = 0.15
    cut_pixel_mad: float = 0.22

    # -------------------------
    # Runtime
    # -------------------------
    pair_batch_size: int = 1
    max_gpus: int = 2
    video_cache_size: int = 8

    # 0 = chạy toàn bộ; đặt 1–10 để smoke test.
    limit_tasks: int = 0

    # Dừng ngay nếu thiếu video.
    fail_on_missing_video: bool = True

    prompt: str = (
        "Retrieve the short ordered video clip that best matches "
        "the user's query. Pay special attention to actions, "
        "temporal order, motion direction, state changes, object "
        "interactions, people, objects, spatial relations, and "
        "scene context. The middle frame is the candidate timestamp."
    )

    @property
    def output_json(self) -> Path:
        return self.output_dir / "submission_final_temporal_top5_private.json"

    @property
    def output_zip(self) -> Path:
        return self.output_dir / "submission_final_temporal_top5_private.zip"

    @property
    def output_csv(self) -> Path:
        return self.output_dir / "temporal_top5_scores.csv"


CFG = TemporalRerankConfig()

print(CFG)

## 3. OOP pipeline

In [ ]:
from __future__ import annotations

from transformers import BitsAndBytesConfig
import csv
import gc
import json
import math
import zipfile
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import cv2
import numpy as np
import torch
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm


# ============================================================
# Data structures
# ============================================================

@dataclass(frozen=True)
class PreparedData:
    predictions: list[dict[str, Any]]
    task_queries: dict[str, str]
    v3c_root: Path
    video_paths: dict[str, Path]


@dataclass(frozen=True)
class GatingDecision:
    winner_index: int
    runner_up_index: int | None
    final_top5_order: list[int]
    promoted: bool
    winner_score: float | None
    old_top1_score: float | None
    runner_up_score: float | None
    margin_over_old_top1: float | None
    margin_over_runner_up: float | None


@dataclass(frozen=True)
class ShardResult:
    predictions: list[dict[str, Any]]
    diagnostics: list[dict[str, Any]]
    promoted_count: int


# ============================================================
# Input repository and path resolution
# ============================================================

class TempoRunRepository:
    def __init__(self, config: TemporalRerankConfig):
        self.config = config

    @staticmethod
    def _load_json(path: Path) -> Any:
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)

    def load_predictions(self) -> list[dict[str, Any]]:
        obj = self._load_json(self.config.input_submission)

        if isinstance(obj, dict) and isinstance(obj.get("predictions"), list):
            predictions = obj["predictions"]
        elif isinstance(obj, list):
            predictions = obj
        else:
            raise ValueError(
                f"Unsupported submission format: {self.config.input_submission}"
            )

        if self.config.limit_tasks > 0:
            predictions = predictions[: self.config.limit_tasks]

        return predictions

    def resolve_tasks_path(self) -> Path:
        for path in self.config.tasks_jsonl_candidates:
            if path.is_file():
                return path

        checked = "\n".join(
            str(path) for path in self.config.tasks_jsonl_candidates
        )
        raise FileNotFoundError(
            "Không tìm thấy public_round_tasks.jsonl. Đã kiểm tra:\n"
            + checked
        )

    @staticmethod
    def load_task_queries(path: Path) -> dict[str, str]:
        queries: dict[str, str] = {}

        with path.open("r", encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue

                row = json.loads(line)
                task_id = row.get("task_id") or row.get("id")
                query = (
                    row.get("description")
                    or row.get("query")
                    or row.get("text")
                )

                if task_id is not None and query:
                    queries[str(task_id)] = str(query)

        return queries

    def resolve_v3c_root(self) -> Path:
        root = self.config.video_dataset_root

        candidates = [
            root,
            root / "V3C TempoRun",
            root / "V3C_TempoRun",
        ]

        if root.is_dir():
            candidates.extend(
                child
                for child in root.iterdir()
                if child.is_dir()
            )

        for candidate in candidates:
            has_v3c1 = (candidate / "V3C1" / "videos").is_dir()
            has_v3c2 = (candidate / "V3C2" / "videos").is_dir()

            if has_v3c1 and has_v3c2:
                return candidate

        raise FileNotFoundError(
            "Không tìm thấy V3C1/videos và V3C2/videos trong "
            f"{root}"
        )

    @staticmethod
    def split_video_id(video_id: str) -> tuple[str, str]:
        parts = str(video_id).split("_", 1)

        if len(parts) != 2:
            raise ValueError(f"video_id không hợp lệ: {video_id}")

        collection = parts[0].upper()
        number = parts[1]

        if collection not in {"V3C1", "V3C2"}:
            raise ValueError(f"Collection không hợp lệ: {video_id}")

        return collection, number

    def build_video_path_map(
        self,
        predictions: list[dict[str, Any]],
        v3c_root: Path,
    ) -> tuple[dict[str, Path], list[str]]:
        needed_video_ids = {
            str(row["video_id"])
            for prediction in predictions
            for row in prediction.get("results", [])[
                : self.config.top_k
            ]
        }

        resolved: dict[str, Path] = {}
        unresolved: set[str] = set()

        for video_id in sorted(needed_video_ids):
            collection, number = self.split_video_id(video_id)

            direct_path = (
                v3c_root
                / collection
                / "videos"
                / number
                / f"{number}.mp4"
            )

            if direct_path.is_file():
                resolved[video_id] = direct_path
            else:
                unresolved.add(video_id)

        # Fallback: scan dataset đúng một lần khi layout khác chuẩn.
        if unresolved:
            wanted: dict[tuple[str, str], str] = {}

            for video_id in unresolved:
                collection, number = self.split_video_id(video_id)
                wanted[(collection, number)] = video_id

            for path in v3c_root.rglob("*.mp4"):
                upper_parts = {part.upper() for part in path.parts}

                if "V3C1" in upper_parts:
                    collection = "V3C1"
                elif "V3C2" in upper_parts:
                    collection = "V3C2"
                else:
                    continue

                key = (collection, path.stem)

                if key in wanted:
                    resolved[wanted[key]] = path

        missing = sorted(needed_video_ids - set(resolved))
        return resolved, missing

    def prepare(self) -> PreparedData:
        config = self.config

        if not config.input_submission.is_file():
            raise FileNotFoundError(config.input_submission)

        if not config.video_dataset_root.is_dir():
            raise FileNotFoundError(config.video_dataset_root)

        if not config.reranker_model.is_dir():
            raise FileNotFoundError(config.reranker_model)

        predictions = self.load_predictions()
        tasks_path = self.resolve_tasks_path()
        task_queries = self.load_task_queries(tasks_path)
        v3c_root = self.resolve_v3c_root()

        video_paths, missing_videos = self.build_video_path_map(
            predictions,
            v3c_root,
        )

        missing_queries = [
            str(prediction["task_id"])
            for prediction in predictions
            if str(prediction["task_id"]) not in task_queries
        ]

        if missing_queries:
            raise KeyError(
                "Thiếu query cho các task: "
                + ", ".join(missing_queries[:20])
            )

        print("INPUT_SUBMISSION :", config.input_submission)
        print("TASKS_JSONL      :", tasks_path)
        print("V3C_ROOT         :", v3c_root)
        print("RERANKER_MODEL   :", config.reranker_model)
        print("TASK COUNT       :", len(predictions))
        print("RESOLVED VIDEOS  :", len(video_paths))
        print("MISSING VIDEOS   :", len(missing_videos))

        if missing_videos:
            print("First missing    :", missing_videos[:20])

            if config.fail_on_missing_video:
                raise FileNotFoundError(
                    f"Còn {len(missing_videos)} video chưa resolve."
                )

        if video_paths:
            sample_id = next(iter(video_paths))
            print(
                "Sample video     :",
                sample_id,
                "->",
                video_paths[sample_id],
            )

        return PreparedData(
            predictions=predictions,
            task_queries=task_queries,
            v3c_root=v3c_root,
            video_paths=video_paths,
        )


# ============================================================
# Per-worker OpenCV cache
# ============================================================

class VideoCaptureCache:
    def __init__(self, max_open: int):
        self.max_open = max_open
        self._items: OrderedDict[
            str,
            tuple[cv2.VideoCapture, float, int],
        ] = OrderedDict()

    def get(
        self,
        path: Path,
    ) -> tuple[cv2.VideoCapture, float, int]:
        key = str(path)

        if key in self._items:
            cap, fps, frame_count = self._items.pop(key)
            self._items[key] = (cap, fps, frame_count)
            return cap, fps, frame_count

        cap = cv2.VideoCapture(key)

        if not cap.isOpened():
            cap.release()
            raise RuntimeError(f"OpenCV không mở được video: {path}")

        fps = float(cap.get(cv2.CAP_PROP_FPS))

        if not math.isfinite(fps) or fps <= 0:
            fps = 12.0

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        self._items[key] = (cap, fps, frame_count)

        while len(self._items) > self.max_open:
            _, (old_cap, _, _) = self._items.popitem(last=False)
            old_cap.release()

        return cap, fps, frame_count

    def close(self) -> None:
        for cap, _, _ in self._items.values():
            cap.release()

        self._items.clear()

    def __enter__(self) -> "VideoCaptureCache":
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> None:
        self.close()


# ============================================================
# Temporal clip construction
# ============================================================

class TemporalClipBuilder:
    def __init__(self, config: TemporalRerankConfig):
        self.config = config

    def _resize_rgb(self, frame: np.ndarray) -> np.ndarray:
        height, width = frame.shape[:2]
        longest = max(height, width)

        if longest <= self.config.max_frame_side:
            return np.ascontiguousarray(frame)

        scale = self.config.max_frame_side / float(longest)
        new_width = max(2, int(round(width * scale)))
        new_height = max(2, int(round(height * scale)))

        resized = cv2.resize(
            frame,
            (new_width, new_height),
            interpolation=cv2.INTER_AREA,
        )

        return np.ascontiguousarray(resized)

    def _read_frame_at_ms(
        self,
        cache: VideoCaptureCache,
        video_path: Path,
        timestamp_ms: float,
    ) -> tuple[np.ndarray, float, float, int]:
        cap, fps, frame_count = cache.get(video_path)

        target_index = max(
            0,
            int(round(timestamp_ms * fps / 1000.0)),
        )

        if frame_count > 0:
            target_index = min(target_index, frame_count - 1)

        candidate_indices = (
            target_index,
            target_index - 1,
            target_index + 1,
            target_index - 2,
            target_index + 2,
        )

        for frame_index in candidate_indices:
            if frame_index < 0:
                continue

            if frame_count > 0 and frame_index >= frame_count:
                continue

            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ok, bgr = cap.read()

            if not ok or bgr is None:
                continue

            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb = self._resize_rgb(rgb)

            actual_ms = frame_index * 1000.0 / fps
            return rgb, actual_ms, fps, frame_count

        raise RuntimeError(
            f"Không đọc được frame gần {timestamp_ms:.1f} ms: "
            f"{video_path}"
        )

    def _is_probable_shot_cut(
        self,
        left_rgb: np.ndarray,
        right_rgb: np.ndarray,
    ) -> bool:
        comparison_size = (160, 90)

        left = cv2.resize(
            left_rgb,
            comparison_size,
            interpolation=cv2.INTER_AREA,
        )
        right = cv2.resize(
            right_rgb,
            comparison_size,
            interpolation=cv2.INTER_AREA,
        )

        left_gray = cv2.cvtColor(
            left,
            cv2.COLOR_RGB2GRAY,
        ).astype(np.float32)

        right_gray = cv2.cvtColor(
            right,
            cv2.COLOR_RGB2GRAY,
        ).astype(np.float32)

        pixel_mad = float(
            np.mean(np.abs(left_gray - right_gray)) / 255.0
        )

        left_hsv = cv2.cvtColor(left, cv2.COLOR_RGB2HSV)
        right_hsv = cv2.cvtColor(right, cv2.COLOR_RGB2HSV)

        left_hist = cv2.calcHist(
            [left_hsv],
            [0, 1],
            None,
            [32, 32],
            [0, 180, 0, 256],
        )
        right_hist = cv2.calcHist(
            [right_hsv],
            [0, 1],
            None,
            [32, 32],
            [0, 180, 0, 256],
        )

        cv2.normalize(left_hist, left_hist)
        cv2.normalize(right_hist, right_hist)

        hist_correlation = float(
            cv2.compareHist(
                left_hist,
                right_hist,
                cv2.HISTCMP_CORREL,
            )
        )

        return (
            hist_correlation < self.config.cut_hist_correlation
            and pixel_mad > self.config.cut_pixel_mad
        )

    def build(
        self,
        cache: VideoCaptureCache,
        video_path: Path,
        center_ms: int,
    ) -> tuple[np.ndarray, dict[str, Any]]:
        _, source_fps, frame_count = cache.get(video_path)

        if frame_count > 0:
            last_valid_ms = (
                (frame_count - 1)
                * 1000.0
                / source_fps
            )
        else:
            last_valid_ms = float("inf")

        frames: list[np.ndarray] = []
        actual_timestamps: list[float] = []

        for offset_ms in self.config.temporal_offsets_ms:
            target_ms = max(
                0.0,
                float(center_ms + offset_ms),
            )

            if math.isfinite(last_valid_ms):
                target_ms = min(target_ms, last_valid_ms)

            frame, actual_ms, _, _ = self._read_frame_at_ms(
                cache,
                video_path,
                target_ms,
            )

            frames.append(frame)
            actual_timestamps.append(actual_ms)

        cut_left = False
        cut_right = False

        if self.config.enable_cut_guard:
            center_index = self.config.temporal_offsets_ms.index(0)

            for index in range(center_index - 1, -1, -1):
                if self._is_probable_shot_cut(
                    frames[index],
                    frames[index + 1],
                ):
                    cut_left = True
                    replacement_frame = frames[index + 1]
                    replacement_ms = actual_timestamps[index + 1]

                    for replace_index in range(index + 1):
                        frames[replace_index] = replacement_frame.copy()
                        actual_timestamps[replace_index] = replacement_ms

                    break

            for index in range(center_index, len(frames) - 1):
                if self._is_probable_shot_cut(
                    frames[index],
                    frames[index + 1],
                ):
                    cut_right = True
                    replacement_frame = frames[index]
                    replacement_ms = actual_timestamps[index]

                    for replace_index in range(
                        index + 1,
                        len(frames),
                    ):
                        frames[replace_index] = replacement_frame.copy()
                        actual_timestamps[replace_index] = replacement_ms

                    break

        clip = np.stack(frames, axis=0)
        clip = np.ascontiguousarray(clip, dtype=np.uint8)

        diagnostic = {
            "source_fps": source_fps,
            "actual_timestamps": [
                round(value, 3)
                for value in actual_timestamps
            ],
            "cut_left": cut_left,
            "cut_right": cut_right,
            "clip_shape": list(clip.shape),
        }

        return clip, diagnostic


# ============================================================
# Qwen temporal scorer
# ============================================================

class QwenTemporalScorer:
    def __init__(
        self,
        config: TemporalRerankConfig,
        gpu_id: int,
    ):
        self.config = config
        self.gpu_id = gpu_id
        self.device = f"cuda:{gpu_id}"
        self.model = self._load_model()

    def _load_model(self) -> CrossEncoder:
        import builtins
    
        print(
            f"[GPU {self.gpu_id}] Loading "
            "Qwen3-VL-Reranker...",
            flush=True,
        )
    
        # Kaggle input là read-only nên không thể ghi lại modules.json.
        # Tạm thời đổi encoding từ utf8 sang utf-8-sig khi thư viện
        # sentence-transformers đọc modules.json.
        original_open = builtins.open


        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
        )
    
        def open_with_bom_support(file, *args, **kwargs):
            if isinstance(file, (str, Path)):
                path = Path(file)
    
                if (
                    path.name == "modules.json"
                    and kwargs.get("encoding") in {"utf8", "utf-8"}
                ):
                    kwargs["encoding"] = "utf-8-sig"
    
            return original_open(file, *args, **kwargs)
    
        try:
            builtins.open = open_with_bom_support
    
            with torch.cuda.device(self.gpu_id):
                model = CrossEncoder(
                    str(self.config.reranker_model),
                    trust_remote_code=True,
                    model_kwargs={
                        "torch_dtype": torch.float16,
                        "attn_implementation": "sdpa",
                        "quantization_config": quantization_config,
                    
                        # Mỗi scorer chỉ nằm trên GPU được chỉ định.
                        "device_map": {"": self.gpu_id},
                    },
                    device=self.device,
                )
    
        finally:
            # Luôn khôi phục open gốc, kể cả khi model load lỗi.
            builtins.open = original_open
    
        print(
            f"[GPU {self.gpu_id}] modalities:",
            model.modalities,
        )
    
        if not model.supports("video"):
            raise RuntimeError(
                f"Model GPU {self.gpu_id} không hỗ trợ video."
            )
    
        return model

    def score(
        self,
        query: str,
        clips: list[np.ndarray],
    ) -> np.ndarray:
        if not clips:
            return np.empty((0,), dtype=np.float32)

        pairs = [(query, clip) for clip in clips]

        scores = self.model.predict(
            pairs,
            batch_size=self.config.pair_batch_size,
            prompt=self.config.prompt,
            show_progress_bar=False,
            activation_fn=torch.nn.Identity(),
            processing_kwargs={
                "video": {
                    "do_sample_frames": False,
                }
            },
        )

        return np.asarray(
            scores,
            dtype=np.float32,
        ).reshape(-1)


# ============================================================
# Conservative top-1 gating
# ============================================================

class ConservativeTop1Policy:
    def __init__(self, config: TemporalRerankConfig):
        self.config = config

    def decide(self, scores: list[float]) -> GatingDecision:
        if not scores:
            return GatingDecision(
                winner_index=0,
                runner_up_index=None,
                final_top5_order=[],
                promoted=False,
                winner_score=None,
                old_top1_score=None,
                runner_up_score=None,
                margin_over_old_top1=None,
                margin_over_runner_up=None,
            )

        valid_indices = [
            index
            for index, score in enumerate(scores)
            if math.isfinite(score)
        ]

        if not valid_indices:
            return GatingDecision(
                winner_index=0,
                runner_up_index=None,
                final_top5_order=list(range(len(scores))),
                promoted=False,
                winner_score=None,
                old_top1_score=None,
                runner_up_score=None,
                margin_over_old_top1=None,
                margin_over_runner_up=None,
            )

        score_order = sorted(
            valid_indices,
            key=lambda index: (
                -scores[index],
                index,
            ),
        )

        winner_index = score_order[0]
        runner_up_index = (
            score_order[1]
            if len(score_order) >= 2
            else None
        )

        winner_score = float(scores[winner_index])
        old_top1_score = (
            float(scores[0])
            if math.isfinite(scores[0])
            else None
        )

        runner_up_score = (
            float(scores[runner_up_index])
            if runner_up_index is not None
            else None
        )

        margin_over_old_top1 = (
            winner_score - old_top1_score
            if old_top1_score is not None
            else None
        )

        margin_over_runner_up = (
            winner_score - runner_up_score
            if runner_up_score is not None
            else None
        )

        promoted = bool(
            winner_index != 0
            and old_top1_score is not None
            and runner_up_score is not None
            and margin_over_old_top1
            >= self.config.promote_over_old_top1
            and margin_over_runner_up
            >= self.config.winner_over_runner_up
        )

        if promoted:
            final_order = [
                winner_index,
                *[
                    index
                    for index in range(len(scores))
                    if index != winner_index
                ],
            ]
        else:
            final_order = list(range(len(scores)))

        return GatingDecision(
            winner_index=winner_index,
            runner_up_index=runner_up_index,
            final_top5_order=final_order,
            promoted=promoted,
            winner_score=winner_score,
            old_top1_score=old_top1_score,
            runner_up_score=runner_up_score,
            margin_over_old_top1=margin_over_old_top1,
            margin_over_runner_up=margin_over_runner_up,
        )


# ============================================================
# Output writer and validation
# ============================================================

class SubmissionWriter:
    def __init__(self, config: TemporalRerankConfig):
        self.config = config

    def validate(
        self,
        input_predictions: list[dict[str, Any]],
        output_predictions: list[dict[str, Any]],
    ) -> None:
        if len(input_predictions) != len(output_predictions):
            raise RuntimeError(
                "Task count mismatch: "
                f"{len(output_predictions)} != {len(input_predictions)}"
            )

        expected_ids = [
            str(prediction["task_id"])
            for prediction in input_predictions
        ]
        actual_ids = [
            str(prediction["task_id"])
            for prediction in output_predictions
        ]

        if expected_ids != actual_ids:
            raise RuntimeError(
                "Output task order hoặc task IDs không khớp input."
            )

        for prediction in output_predictions:
            rows = prediction.get("results", [])

            if not rows:
                raise RuntimeError(
                    f"{prediction['task_id']}: output rỗng."
                )

            if len(rows) > self.config.final_top:
                raise RuntimeError(
                    f"{prediction['task_id']}: quá "
                    f"{self.config.final_top} results."
                )

            seen: set[tuple[str, int]] = set()

            for expected_rank, row in enumerate(rows, start=1):
                if int(row["rank"]) != expected_rank:
                    raise RuntimeError(
                        f"{prediction['task_id']}: rank không liên tục."
                    )

                key = (
                    str(row["video_id"]),
                    int(row["frame_ms"]),
                )

                if key in seen:
                    raise RuntimeError(
                        f"{prediction['task_id']}: duplicate frame {key}."
                    )

                seen.add(key)

    def write(
        self,
        input_predictions: list[dict[str, Any]],
        output_predictions: list[dict[str, Any]],
        diagnostics: list[dict[str, Any]],
    ) -> None:
        config = self.config
        config.output_dir.mkdir(parents=True, exist_ok=True)

        self.validate(input_predictions, output_predictions)

        with config.output_json.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                {"predictions": output_predictions},
                file,
                ensure_ascii=False,
                separators=(",", ":"),
            )

        if config.output_zip.exists():
            config.output_zip.unlink()

        with zipfile.ZipFile(
            config.output_zip,
            "w",
            compression=zipfile.ZIP_DEFLATED,
        ) as zip_file:
            zip_file.write(
                config.output_json,
                arcname="submission.json",
            )

        if diagnostics:
            with config.output_csv.open(
                "w",
                encoding="utf-8",
                newline="",
            ) as file:
                writer = csv.DictWriter(
                    file,
                    fieldnames=list(diagnostics[0].keys()),
                )
                writer.writeheader()
                writer.writerows(diagnostics)


# ============================================================
# Main OOP pipeline
# ============================================================

class TemporalTop5RerankPipeline:
    def __init__(self, config: TemporalRerankConfig):
        self.config = config
        self.repository = TempoRunRepository(config)
        self.clip_builder = TemporalClipBuilder(config)
        self.policy = ConservativeTop1Policy(config)
        self.writer = SubmissionWriter(config)

        self.data: PreparedData | None = None
        self.scorers: dict[int, QwenTemporalScorer] = {}

    @staticmethod
    def _normalize_rows(
        rows: Iterable[dict[str, Any]],
        final_top: int,
    ) -> list[dict[str, Any]]:
        normalized: list[dict[str, Any]] = []

        for position, row in enumerate(
            list(rows)[:final_top],
            start=1,
        ):
            normalized.append(
                {
                    "rank": position,
                    "video_id": str(row["video_id"]),
                    "frame_ms": int(row["frame_ms"]),
                }
            )

        return normalized

    def _create_final_prediction(
        self,
        task_id: str,
        original_rows: list[dict[str, Any]],
        top5_order: list[int],
    ) -> dict[str, Any]:
        config = self.config

        original_top5 = original_rows[: config.top_k]
        original_tail = original_rows[
            config.top_k : config.final_top
        ]

        ordered_rows = (
            [original_top5[index] for index in top5_order]
            + original_tail
        )

        final_rows: list[dict[str, Any]] = []
        seen_frames: set[tuple[str, int]] = set()

        for row in ordered_rows:
            key = (
                str(row["video_id"]),
                int(row["frame_ms"]),
            )

            if key in seen_frames:
                continue

            seen_frames.add(key)

            final_rows.append(
                {
                    "rank": len(final_rows) + 1,
                    "video_id": key[0],
                    "frame_ms": key[1],
                }
            )

            if len(final_rows) >= config.final_top:
                break

        return {
            "task_id": task_id,
            "results": final_rows,
        }

    def _process_task(
        self,
        scorer: QwenTemporalScorer,
        cache: VideoCaptureCache,
        prediction: dict[str, Any],
    ) -> tuple[
        dict[str, Any],
        list[dict[str, Any]],
        bool,
    ]:
        assert self.data is not None
    
        config = self.config
        task_id = str(prediction["task_id"])
        query = self.data.task_queries[task_id]
    
        original_rows = self._normalize_rows(
            prediction.get("results", []),
            config.final_top,
        )
        top5 = original_rows[: config.top_k]
    
        # Score cuối của mỗi candidate:
        #
        # max_shift(
        #     raw_score
        #     - distance_penalty * abs(shift_ms) / 1000
        # )
        temporal_scores = [
            float("-inf")
            for _ in top5
        ]
    
        best_raw_scores: list[float] = [
            float("-inf")
            for _ in top5
        ]
    
        best_shift_ms: list[int | None] = [
            None
            for _ in top5
        ]
    
        best_shifted_center_ms: list[int | None] = [
            None
            for _ in top5
        ]
    
        # Tất cả clip của toàn bộ top-5 được gom lại
        # để gọi scorer.score() một lần.
        clips: list[np.ndarray] = []
    
        clip_metadata: list[dict[str, Any]] = []
    
        diagnostics_by_position: dict[
            int,
            dict[str, Any],
        ] = {}
    
        local_details_by_position: dict[
            int,
            list[dict[str, Any]],
        ] = {
            position: []
            for position in range(len(top5))
        }
    
        # --------------------------------------------------------
        # Tạo:
        #
        # top-5 candidates
        # × 5 local shifts
        # = tối đa 25 clip
        #
        # Mỗi clip vẫn sử dụng 11 temporal_offsets_ms hiện tại.
        # --------------------------------------------------------
        for position, row in enumerate(top5):
            video_path = self.data.video_paths.get(
                row["video_id"]
            )
    
            original_frame_ms = int(row["frame_ms"])
    
            if video_path is None:
                diagnostics_by_position[position] = {
                    "video_path": "",
                    "error": "video_not_found",
                }
                continue
    
            diagnostics_by_position[position] = {
                "video_path": str(video_path),
                "error": "",
            }
    
            for shift_ms in config.local_shifts_ms:
                shifted_center_ms = max(
                    0,
                    original_frame_ms + int(shift_ms),
                )
    
                penalty = (
                    config.distance_penalty
                    * abs(int(shift_ms))
                    / 1000.0
                )
    
                try:
                    clip, diagnostic = self.clip_builder.build(
                        cache=cache,
                        video_path=video_path,
                        center_ms=shifted_center_ms,
                    )
    
                    clips.append(clip)
    
                    clip_metadata.append(
                        {
                            "position": position,
                            "shift_ms": int(shift_ms),
                            "shifted_center_ms": shifted_center_ms,
                            "penalty": float(penalty),
                            "diagnostic": diagnostic,
                            "video_path": str(video_path),
                        }
                    )
    
                except Exception as exception:
                    local_details_by_position[position].append(
                        {
                            "shift_ms": int(shift_ms),
                            "shifted_center_ms": shifted_center_ms,
                            "raw_score": None,
                            "penalty": float(penalty),
                            "adjusted_score": None,
                            "error": (
                                f"{type(exception).__name__}: "
                                f"{exception}"
                            ),
                        }
                    )
    
        # --------------------------------------------------------
        # Chấm tất cả clip.
        # --------------------------------------------------------
        if clips:
            predicted_scores = scorer.score(
                query=query,
                clips=clips,
            )
    
            if len(predicted_scores) != len(clip_metadata):
                raise RuntimeError(
                    f"{task_id}: score count mismatch "
                    f"{len(predicted_scores)} != "
                    f"{len(clip_metadata)}"
                )
    
            for metadata, score in zip(
                clip_metadata,
                predicted_scores,
            ):
                position = int(metadata["position"])
                shift_ms = int(metadata["shift_ms"])
                shifted_center_ms = int(
                    metadata["shifted_center_ms"]
                )
                penalty = float(metadata["penalty"])
    
                raw_score = float(score)
    
                if math.isfinite(raw_score):
                    adjusted_score = raw_score - penalty
                else:
                    adjusted_score = float("-inf")
    
                local_details_by_position[position].append(
                    {
                        "shift_ms": shift_ms,
                        "shifted_center_ms": shifted_center_ms,
                        "raw_score": (
                            raw_score
                            if math.isfinite(raw_score)
                            else None
                        ),
                        "penalty": penalty,
                        "adjusted_score": (
                            adjusted_score
                            if math.isfinite(adjusted_score)
                            else None
                        ),
                        "error": (
                            ""
                            if math.isfinite(raw_score)
                            else "non_finite_score"
                        ),
                    }
                )
    
                if not math.isfinite(adjusted_score):
                    continue
    
                current_best = temporal_scores[position]
                current_shift = best_shift_ms[position]
    
                is_better_score = (
                    adjusted_score > current_best
                )
    
                # Nếu adjusted score bằng nhau,
                # ưu tiên shift gần 0 hơn.
                is_better_tie = (
                    math.isclose(
                        adjusted_score,
                        current_best,
                        rel_tol=0.0,
                        abs_tol=1e-8,
                    )
                    and (
                        current_shift is None
                        or abs(shift_ms) < abs(current_shift)
                    )
                )
    
                if is_better_score or is_better_tie:
                    temporal_scores[position] = (
                        adjusted_score
                    )
                    best_raw_scores[position] = raw_score
                    best_shift_ms[position] = shift_ms
                    best_shifted_center_ms[position] = (
                        shifted_center_ms
                    )
    
                    diagnostic = metadata["diagnostic"]
    
                    diagnostics_by_position[position] = {
                        **diagnostic,
                        "video_path": metadata["video_path"],
                        "error": "",
                    }
    
        # Score truyền vào policy bây giờ là:
        #
        # best adjusted score của từng candidate.
        decision = self.policy.decide(
            temporal_scores
        )
    
        # Hàm này chỉ thay đổi thứ tự original_rows.
        # Vì vậy frame_ms output vẫn là frame_ms ban đầu,
        # không phải best_shifted_center_ms.
        final_prediction = self._create_final_prediction(
            task_id=task_id,
            original_rows=original_rows,
            top5_order=decision.final_top5_order,
        )
    
        diagnostic_rows: list[dict[str, Any]] = []
    
        for position, row in enumerate(top5):
            diagnostic = diagnostics_by_position.get(
                position,
                {},
            )
    
            adjusted_score = temporal_scores[position]
            raw_score = best_raw_scores[position]
    
            local_details = sorted(
                local_details_by_position[position],
                key=lambda item: int(item["shift_ms"]),
            )
    
            diagnostic_rows.append(
                {
                    "task_id": task_id,
                    "original_rank": position + 1,
                    "new_rank": (
                        decision.final_top5_order.index(position)
                        + 1
                    ),
                    "video_id": row["video_id"],
    
                    # Timestamp output vẫn giữ nguyên.
                    "frame_ms": row["frame_ms"],
    
                    # temporal_score bây giờ là adjusted score tốt nhất.
                    "temporal_score": (
                        adjusted_score
                        if math.isfinite(adjusted_score)
                        else None
                    ),
    
                    "best_raw_score": (
                        raw_score
                        if math.isfinite(raw_score)
                        else None
                    ),
    
                    "best_shift_ms": best_shift_ms[position],
    
                    # Chỉ diagnostic, không dùng làm output frame_ms.
                    "best_shifted_center_ms": (
                        best_shifted_center_ms[position]
                    ),
    
                    "distance_penalty": (
                        config.distance_penalty
                    ),
    
                    "local_shift_scores": json.dumps(
                        local_details,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),
    
                    "temporal_winner": (
                        position == decision.winner_index
                        and decision.winner_score is not None
                    ),
    
                    "promoted_to_rank1": (
                        decision.promoted
                        and position == decision.winner_index
                    ),
    
                    "winner_score": decision.winner_score,
                    "old_top1_score": decision.old_top1_score,
                    "runner_up_score": decision.runner_up_score,
    
                    "margin_over_old_top1": (
                        decision.margin_over_old_top1
                    ),
    
                    "margin_over_runner_up": (
                        decision.margin_over_runner_up
                    ),
    
                    "video_path": diagnostic.get(
                        "video_path",
                        "",
                    ),
    
                    "source_fps": diagnostic.get(
                        "source_fps"
                    ),
    
                    # Timestamp của clip thuộc shift tốt nhất.
                    "actual_timestamps_ms": json.dumps(
                        diagnostic.get(
                            "actual_timestamps",
                            [],
                        )
                    ),
    
                    "cut_left": diagnostic.get(
                        "cut_left"
                    ),
    
                    "cut_right": diagnostic.get(
                        "cut_right"
                    ),
    
                    "clip_shape": json.dumps(
                        diagnostic.get(
                            "clip_shape",
                            [],
                        )
                    ),
    
                    "error": diagnostic.get(
                        "error",
                        "",
                    ),
                }
            )
    
        return (
            final_prediction,
            diagnostic_rows,
            decision.promoted,
        )

    def _run_shard(
        self,
        shard_index: int,
        gpu_id: int,
        shard_predictions: list[dict[str, Any]],
    ) -> ShardResult:
        scorer = self.scorers[gpu_id]
        torch.cuda.set_device(gpu_id)

        output_predictions: list[dict[str, Any]] = []
        output_diagnostics: list[dict[str, Any]] = []
        promoted_count = 0

        with VideoCaptureCache(
            max_open=self.config.video_cache_size
        ) as cache:
            with torch.cuda.device(gpu_id):
                for prediction in tqdm(
                    shard_predictions,
                    desc=f"GPU {gpu_id}: temporal top5",
                    position=shard_index,
                    leave=True,
                ):
                    (
                        final_prediction,
                        diagnostics,
                        promoted,
                    ) = self._process_task(
                        scorer=scorer,
                        cache=cache,
                        prediction=prediction,
                    )

                    output_predictions.append(final_prediction)
                    output_diagnostics.extend(diagnostics)
                    promoted_count += int(promoted)

        return ShardResult(
            predictions=output_predictions,
            diagnostics=output_diagnostics,
            promoted_count=promoted_count,
        )

    def _load_scorers(self, gpu_ids: list[int]) -> None:
        # Load tuần tự để tránh nhiều worker cùng đọc checkpoint.
        for gpu_id in gpu_ids:
            self.scorers[gpu_id] = QwenTemporalScorer(
                config=self.config,
                gpu_id=gpu_id,
            )

    def run(self) -> None:
        if not torch.cuda.is_available():
            raise RuntimeError("Cần CUDA GPU để chạy notebook này.")

        self.data = self.repository.prepare()

        gpu_count = min(
            self.config.max_gpus,
            torch.cuda.device_count(),
        )

        if gpu_count <= 0:
            raise RuntimeError("Không có GPU khả dụng.")

        gpu_ids = list(range(gpu_count))
        print("GPU IDs           :", gpu_ids)

        self._load_scorers(gpu_ids)

        prediction_shards = [
            [
                prediction
                for index, prediction
                in enumerate(self.data.predictions)
                if index % gpu_count == shard_index
            ]
            for shard_index in range(gpu_count)
        ]

        print(
            "Shard sizes       :",
            [len(shard) for shard in prediction_shards],
        )

        with ThreadPoolExecutor(
            max_workers=gpu_count,
        ) as executor:
            futures = [
                executor.submit(
                    self._run_shard,
                    shard_index,
                    gpu_id,
                    prediction_shards[shard_index],
                )
                for shard_index, gpu_id in enumerate(gpu_ids)
            ]

            shard_results = [
                future.result()
                for future in futures
            ]

        prediction_by_task: dict[
            str,
            dict[str, Any],
        ] = {}

        all_diagnostics: list[dict[str, Any]] = []
        total_promoted = 0

        for shard_result in shard_results:
            for prediction in shard_result.predictions:
                prediction_by_task[
                    str(prediction["task_id"])
                ] = prediction

            all_diagnostics.extend(shard_result.diagnostics)
            total_promoted += shard_result.promoted_count

        final_predictions = [
            prediction_by_task[str(prediction["task_id"])]
            for prediction in self.data.predictions
        ]

        task_order = {
            str(prediction["task_id"]): index
            for index, prediction
            in enumerate(self.data.predictions)
        }

        all_diagnostics.sort(
            key=lambda row: (
                task_order[str(row["task_id"])],
                int(row["original_rank"]),
            )
        )

        self.writer.write(
            input_predictions=self.data.predictions,
            output_predictions=final_predictions,
            diagnostics=all_diagnostics,
        )

        print()
        print("================ DONE ================")
        print("Tasks              :", len(final_predictions))
        print(
            "Promoted to rank 1 :",
            total_promoted,
        )
        print("JSON               :", self.config.output_json)
        print("ZIP                :", self.config.output_zip)
        print("CSV                :", self.config.output_csv)
        print("======================================")

    def release(self) -> None:
        self.scorers.clear()
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 4. Chạy pipeline

In [ ]:
pipeline = TemporalTop5RerankPipeline(CFG)

try:
    pipeline.run()
finally:
    # Giải phóng model sau khi output đã được ghi.
    pipeline.release()

## 5. Kiểm tra output

In [ ]:
import json
import zipfile

with CFG.output_json.open("r", encoding="utf-8") as file:
    submission = json.load(file)

with zipfile.ZipFile(CFG.output_zip, "r") as zip_file:
    zip_names = zip_file.namelist()

print("Predictions:", len(submission["predictions"]))
print("ZIP content:", zip_names)
print("First task :", submission["predictions"][0])
print("Submit file:", CFG.output_zip)